https://www.kaggle.com/datasets/yusufmurtaza01/tomato-leaf-disease

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import os

from PIL import Image

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
import torch
from torch import nn
from torch.optim import Adam
from torchvision.transforms import transforms
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary

try :
    import torch_directml
    device = torch_directml.device(0) 
except:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

In [ ]:
image_path = []
labels = []

for label in os.listdir('img'):   
    for image in os.listdir(f'img/{label}'):
        image_path.append(f'img/{label}/{image}')
        labels.append(label)

df = pd.DataFrame(zip(image_path, labels), columns=['image_path', 'labels'])

print(df['labels'].unique())
display(df.head())

In [ ]:
n_rows = 3
n_cols = 3

f, axarr = plt.subplots(n_rows, n_cols)

for row in range(n_rows):
    for col in range(n_cols):
        image = Image.open(df.sample(n=1)['image_path'].iloc[0]).convert('RGB')
        axarr[row, col].imshow(image)
        axarr[row, col].axis('off')

plt.show()

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=0)
test_df, val_df = train_test_split(test_df, test_size=0.5, random_state=0)

print(f'Train dataset shape : {train_df.shape}')
print(f'Validation dataset shape : {val_df.shape}')
print(f'Test dataset shape : {test_df.shape}')

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(df['labels'])

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float),
])

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transform = None):
        self.dataframe = dataframe
        self.transform = transform
        self.labels = torch.tensor(label_encoder.transform(dataframe['labels'])).to(device)

    def __len__(self):
        return self.dataframe.shape[0]
    
    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image).to(device)

        return image, label

In [ ]:
train_dataset = CustomImageDataset(dataframe= train_df, transform= transform)
val_dataset = CustomImageDataset(dataframe= val_df, transform= transform)
test_dataset = CustomImageDataset(dataframe= test_df, transform= transform)

In [ ]:
LR = 1e-4
BATCH_SIZE = 64
EPOCHS = 12

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pooling = nn.MaxPool2d(2,2)

        self.relu = nn.ReLU()

        self.flatten = nn.Flatten()
        self.linear = nn.Linear((128*16*16), 128) 

        self.output = nn.Linear(128, len(df['labels'].unique()))

    def forward(self, x):
        x = self.conv1(x) 
        x = self.pooling(x)
        x = self.relu(x)

        x = self.conv2(x) 
        x = self.pooling(x) 
        x = self.relu(x)

        x = self.conv3(x) 
        x = self.pooling(x)
        x = self.relu(x)

        x = self.flatten(x)
        x = self.linear(x)
        x = self.output(x)
        return x

In [ ]:
model = Net().to(device)

In [ ]:
try:
    summary(model, input_size = (3, 128, 128))
except:
    pass

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR)

In [ ]:
total_loss_train_plot = []
total_acc_train_plot = []
total_loss_val_plot = []
total_acc_val_plot = []

for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_loss_val = 0
    total_acc_val = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        train_loss = criterion(outputs, labels)
        total_loss_train += train_loss.item()

        train_loss.backward()

        train_acc = (torch.argmax(outputs, axis=1) == labels).sum().item()
        total_acc_train += train_acc

        optimizer.step()

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            val_loss = criterion(outputs, labels)
            total_loss_val += val_loss.item()

            val_acc = (torch.argmax(outputs, axis=1) == labels).sum().item()
            total_acc_val += val_acc

    len_train = len(train_dataset)
    len_val = len(val_dataset)

    total_loss_train_plot.append(round(total_loss_train/len_train, 4))
    total_loss_val_plot.append(round(total_loss_val/len_val, 4))  

    total_acc_train_plot.append(round((total_acc_train / len_train) * 100, 4))
    total_acc_val_plot.append(round((total_acc_val / len_val) * 100, 4))  
    
    print('='*25)
    print(f''' Epoch: {epoch+1}/{EPOCHS}, 
    Train Loss: {round(total_loss_train / len_train, 4)}, Train Accuracy: {round((total_acc_train / len_train) * 100, 4)}%
    Validation Loss: {round(total_loss_val / len_val, 4)}, Validation Accuracy: {round((total_acc_val / len_val) * 100, 4)}%
    ''')

In [ ]:
with torch.no_grad():
    total_loss_test = 0
    total_acc_test = 0

    for inputs, labels in test_loader:
        predictions = model(inputs)

        test_loss = criterion(predictions, labels)
        total_loss_test += test_loss.item()

        total_acc_test += (torch.argmax(predictions, axis=1) == labels).sum().item()

        len_test = len(test_dataset)   
        avg_loss_test = total_loss_test / len_test
        avg_acc_test = (total_acc_test / len_test) * 100

    print(f'''Test Loss: {round(avg_loss_test, 4)}, Test Accuracy: {round(avg_acc_test, 4)}%''')

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(15,5))

ax[0].plot(total_loss_train_plot, label='Training Loss')
ax[0].plot(total_loss_val_plot, label='Validation Loss')
ax[0].set_title('Training and Validation loss over epochs')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_ylim([0, 1])
ax[0].legend()

ax[1].plot(total_acc_train_plot, label='Training Accuracy')
ax[1].plot(total_acc_val_plot, label='Validation Accuracy')
ax[1].set_title('Training and Validation Accuracy over epochs')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy')
ax[1].set_ylim([0, 100])
ax[1].legend()

plt.show()

In [ ]:
def predict_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = transform(image).to(device)

    output = model(image.unsqueeze(0))
    output = torch.argmax(output, axis=1).item()

    return label_encoder.inverse_transform([output])

predict_image('spider_img.jpg')